# Dendrite spine distance methods comparison

This notebook compares four distance models between spine attachment points: baseline cylindrical distance, skeleton stem approximation, shortest path over original mesh edges, and Heat Method geodesic approximation.

In [1]:
from pathlib import Path

import faulthandler
import pandas as pd
import trimesh

from dendrite_analysis import Dendrite, reset_saved_data, set_output_dir, summarize_distance_results

faulthandler.enable()


c:\Users\Student\.conda\envs\spine-analysis\lib\site-packages\libpysal\weights\util.py:23: UserWarning: geopandas not available. Some functionality will be disabled.
  warn("geopandas not available. Some functionality will be disabled.")


In [4]:
DATASET_PATHS = [
    Path("example_dendrite_microns"),
]

OUTPUT_DIR = Path("output_dendrite_distance_comparison/4")
METHODS = (
    "cylinder",
    "stem_graph",
    "mesh_graph",
    # "heat",
)
SPINE_FILE_PATTERN = "spine_*.off"

In [5]:
from dendrite_analysis.mesh_repair import repair_mesh
from dendrite_analysis.surface_distances import polyhedron_to_trimesh
from spine_analysis.mesh.utils import _mesh_to_v_f, write_off


DEBUG_LOG_PATH = Path("dendrite_distance_debug.log")


def log_step(message):
    print(message, flush=True)
    with DEBUG_LOG_PATH.open("a", encoding="utf-8") as fd:
        fd.write(message + "\n")
        fd.flush()


def load_trimesh_dataset(dataset_path, spine_file_pattern="spine_*.off"):
    dataset_path = Path(dataset_path)
    spine_paths = sorted(dataset_path.glob(spine_file_pattern))
    spine_meshes = {}
    dendrite_meshes = {}
    spine_to_dendrite = {}

    for spine_path in spine_paths:
        spine_key = str(spine_path).replace("\\", "/")
        dendrite_path = spine_path.parent / "surface_mesh.off"
        dendrite_key = str(dendrite_path).replace("\\", "/")
        if dendrite_key not in dendrite_meshes:
            dendrite_meshes[dendrite_key] = trimesh.load_mesh(str(dendrite_path), process=False)
        spine_meshes[spine_key] = trimesh.load_mesh(str(spine_path), process=False)
        spine_to_dendrite[spine_key] = dendrite_key

    return spine_meshes, dendrite_meshes, spine_to_dendrite


def save_repaired_mesh(repaired_mesh, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if hasattr(repaired_mesh, "export"):
        repaired_mesh.export(str(output_path))
        return output_path

    vertices, faces = _mesh_to_v_f(repaired_mesh)
    with output_path.open("w") as fd:
        write_off(fd, vertices, faces)
    return output_path


try:
    from tqdm.auto import tqdm
    _tqdm_available = True
except ImportError:
    _tqdm_available = False

all_summaries = []

dataset_iter = tqdm(DATASET_PATHS, desc="Datasets", unit="dataset") if _tqdm_available else DATASET_PATHS
for dataset_path in dataset_iter:
    log_step(f"[dataset] loading {dataset_path}")
    spine_meshes_all, dendrite_meshes, spine_to_dendrite = load_trimesh_dataset(
        dataset_path,
        spine_file_pattern=SPINE_FILE_PATTERN,
    )
    log_step(f"[dataset] loaded {len(dendrite_meshes)} dendrite mesh(es), {len(spine_meshes_all)} spine mesh(es)")

    dendrite_items = list(dendrite_meshes.items())
    dendrite_iter = (
        tqdm(dendrite_items, desc=f"{dataset_path.name} — dendrites", unit="dendrite", leave=False)
        if _tqdm_available
        else dendrite_items
    )

    for dendrite_mesh_name, dendrite_mesh in dendrite_iter:
        log_step(f"[dendrite] start {dendrite_mesh_name}")
        dendrite_mesh_for_repair = polyhedron_to_trimesh(dendrite_mesh)
        log_step(f"[dendrite] repair start {dendrite_mesh_name}")
        repaired_mesh, report_before, report_after = repair_mesh(dendrite_mesh_for_repair, verbose=True)
        log_step(f"[dendrite] repair done {dendrite_mesh_name}")

        print("\nBefore:", report_before)
        print("\nAfter:", report_after)

        dendrite_name = Path(dendrite_mesh_name).stem
        if not _tqdm_available:
            print(f"[{dataset_path.name}] Processing dendrite: {dendrite_name}")
        dendrite_output_dir = OUTPUT_DIR / dataset_path.name / dendrite_name
        repaired_mesh_path = dendrite_output_dir / f"repaired_{Path(dendrite_mesh_name).name}"
        save_repaired_mesh(repaired_mesh, repaired_mesh_path)
        log_step(f"Saved repaired mesh to: {repaired_mesh_path}")

        reset_saved_data()
        set_output_dir(str(dendrite_output_dir / "metrics"))

        spine_meshes = {
            spine_name: spine_mesh
            for spine_name, spine_mesh in spine_meshes_all.items()
            if spine_to_dendrite[spine_name] == dendrite_mesh_name
        }

        log_step(f"[dendrite] Dendrite init start {dendrite_mesh_name} with {len(spine_meshes)} spines")
        dendrite = Dendrite(dendrite_name, {dendrite_mesh_name: repaired_mesh}, spine_meshes)
        log_step(f"[dendrite] Dendrite init done {dendrite_mesh_name}")
        log_step(f"[dendrite] distance methods start {dendrite_mesh_name}: {METHODS}")
        results = dendrite.calculate_spine_distance_matrices(
            methods=METHODS,
            output_dir=str(dendrite_output_dir / "distance_methods"),
        )
        log_step(f"[dendrite] distance methods done {dendrite_mesh_name}")

        summary = summarize_distance_results(results)
        summary.insert(0, "dendrite", dendrite_name)
        summary.insert(0, "dataset", str(dataset_path))
        all_summaries.append(summary)

comparison_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
comparison_summary



Datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

spine_name example_dendrite_microns\spine_0.off
spine_name example_dendrite_microns\spine_1.off
spine_name example_dendrite_microns\spine_10.off
spine_name example_dendrite_microns\spine_11.off
spine_name example_dendrite_microns\spine_12.off
spine_name example_dendrite_microns\spine_13.off
spine_name example_dendrite_microns\spine_14.off
spine_name example_dendrite_microns\spine_15.off
spine_name example_dendrite_microns\spine_16.off
spine_name example_dendrite_microns\spine_17.off
spine_name example_dendrite_microns\spine_18.off
spine_name example_dendrite_microns\spine_19.off
spine_name example_dendrite_microns\spine_2.off
spine_name example_dendrite_microns\spine_3.off
spine_name example_dendrite_microns\spine_4.off
spine_name example_dendrite_microns\spine_5.off
spine_name example_dendrite_microns\spine_6.off
spine_name example_dendrite_microns\spine_7.off
spine_name example_dendrite_microns\spine_8.off
spine_name example_dendrite_microns\spine_9.off

✅ Итог: ошибок чтения шипов н

example_dendrite_microns — dendrites:   0%|          | 0/1 [00:00<?, ?dendrite/s]

=== Before repair ===
Mesh report:
  Vertices              : 2550
  Faces                 : 4987
  Valid (lib check)     : True
  Closed / watertight   : False
  Boundary halfedges    : 121
  Volume negative       : False
  Self-intersections    : True
  Degenerate faces      : 1
  Issues detected       :
    ✗ Mesh has 121 boundary halfedges (60 boundary edges) — open holes exist
    ✗ Mesh is not closed — volume is undefined
    ✗ Mesh has self-intersecting face pairs
    ✗ 1 degenerate (zero-area) faces
  [1/5] CGAL stitch_borders applied.
  [2/5] Removed 1 degenerate faces.
  [3/5] Split 5 non-manifold vertex(es) into separate fans.
  [4/5] Holes filled — mesh is now watertight.
  [5/5] Normals OK (volume: 2973090542.9465).
  [pre-convert] trimesh: watertight, volume = 2973090542.946493
  Converting 2558 vertices / 5104 faces to Polyhedron_3 (may take a moment for large meshes)...

=== After repair ===
Mesh report:
  Vertices              : 2558
  Faces                 : 5104
  Val

: 

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_summary.to_csv(OUTPUT_DIR / "all_distance_methods_summary.csv", index=False)
comparison_summary

,dataset,dendrite,method,elapsed_seconds,n_points,mean_distance,median_distance,max_distance
0,example_dendrite,surface_mesh,cylinder,0.007647,22,14.608155,13.225813,37.092314
1,example_dendrite,surface_mesh,mesh_graph,0.990088,22,20.056489,18.301545,49.464770
2,example_dendrite,surface_mesh,stem_graph,6.505217,22,13.861903,12.923086,32.800905
